# 04 - Multi-Qubit Gates and Controlled Operations

**Prerequisites:** Notebooks 01-03. Familiarity with single-qubit gates and measurement.

Single-qubit gates can create superposition, but they cannot create
**entanglement** -- the defining resource of quantum computing. For that, we
need gates that act on two or more qubits simultaneously. Multi-qubit gates
introduce **conditional logic** into quantum circuits: one qubit's state
controls what happens to another.

By the end of this notebook you will be able to:

1. **Describe** CNOT, CZ, SWAP, Toffoli, and Fredkin gates and their truth tables.
2. **Implement** controlled operations using gate.Controlled and builder.Ctrl.
3. **Verify** CZ symmetry and construct custom unitary gates.
4. **Explain** how multi-qubit gates are decomposed into simpler operations.

In this notebook we will:

1. Master the **CNOT** gate and build its full truth table.
2. Explore **CZ** symmetry -- why swapping control and target gives the same result.
3. Use the **SWAP** gate to exchange qubit states.
4. Study the **Toffoli (CCX)** gate as a reversible AND.
5. Demonstrate the **Fredkin (CSWAP)** gate as a controlled SWAP.
6. Build **controlled-U** and **multi-controlled** gates from the Goqu API.
7. Create **custom unitary** gates from user-defined matrices.
8. Examine **gate decomposition** into primitive operations.
9. Predict and verify the result of applying CNOT to a superposition state.

In [1]:
import (
	"fmt"
	"math"
	"math/cmplx"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## CNOT -- The Fundamental Two-Qubit Gate

The **CNOT** (controlled-NOT, also called CX) gate flips the **target** qubit
if and only if the **control** qubit is |1>. It is the standard entangling
gate in quantum computing and, together with single-qubit gates, forms a
universal gate set.

In our circuit `CNOT(0, 1)`, qubit 0 is the **control** and qubit 1 is the
**target**. Remember the bitstring convention: $|q_1\,q_0\rangle$ where qubit 0
is the **rightmost** bit.

| Input | Output | Explanation |
|:-----:|:------:|:------------|
| \|00⟩ | \|00⟩  | q0=0 (control off), target q1 unchanged |
| \|01⟩ | \|11⟩  | q0=1 (control on), target q1 flipped 0→1 |
| \|10⟩ | \|10⟩  | q0=0 (control off), target q1 unchanged |
| \|11⟩ | \|01⟩  | q0=1 (control on), target q1 flipped 1→0 |

Let's verify this truth table by evolving all four computational basis states.

In [ ]:
%%
// CNOT truth table: evolve all 4 basis states |00>, |01>, |10>, |11>
// Bitstring convention: |q1 q0> where q0 (rightmost) is the control
fmt.Println("CNOT Truth Table  (control=q0, target=q1)")
fmt.Println("==========================================")
fmt.Println("  Input -> Output")

for idx := 0; idx < 4; idx++ {
	b := builder.New("cnot", 2)
	// idx bit k -> qubit k: bit 0 = q0, bit 1 = q1
	if idx&1 == 1 {
		b.X(0)
	}
	if (idx>>1)&1 == 1 {
		b.X(1)
	}
	b.CNOT(0, 1)
	c, _ := b.Build()

	sim := statevector.New(2)
	sim.Evolve(c)
	sv := sim.StateVector()

	for outIdx, amp := range sv {
		if real(amp)*real(amp)+imag(amp)*imag(amp) > 0.5 {
			q0 := idx & 1
			flipped := ""
			if (idx>>1)&1 != (outIdx>>1)&1 {
				flipped = fmt.Sprintf("  <-- q0=%d, target q1 flipped", q0)
			}
			fmt.Printf("  |%02b>   ->  |%02b>%s\n", idx, outIdx, flipped)
		}
	}
}

// Show the circuit diagram for CNOT(0,1) with q0=|1>
fmt.Println("\nCircuit for |01> -> |11>:")
cDemo, _ := builder.New("cnot-demo", 2).X(0).CNOT(0, 1).Build()
gonbui.DisplayHTML(draw.SVG(cDemo))

## CZ Gate and Its Symmetry

The **CZ** (controlled-Z) gate applies a Z gate to the target when the control
is |1>. Its matrix is `diag(1, 1, 1, -1)` -- it flips the phase of |11> to
-|11> and leaves all other basis states unchanged.

A remarkable property of CZ is that it is **symmetric**: swapping the control
and target qubits gives the same result. This is because the matrix is
diagonal, so CZ(0,1) = CZ(1,0). Let's verify this.

In [3]:
%%
// CZ symmetry: CZ(0,1) == CZ(1,0) on the same input
// Prepare |+1> = H(0) X(1) and apply CZ both ways
fmt.Println("CZ Symmetry Demonstration")
fmt.Println("=========================")

// CZ with q0 as control
c01, _ := builder.New("cz-01", 2).H(0).X(1).CZ(0, 1).Build()
sim01 := statevector.New(2)
sim01.Evolve(c01)
sv01 := sim01.StateVector()

// CZ with q1 as control
c10, _ := builder.New("cz-10", 2).H(0).X(1).CZ(1, 0).Build()
sim10 := statevector.New(2)
sim10.Evolve(c10)
sv10 := sim10.StateVector()

fmt.Println("Input state: H|0> tensor X|0> = |+1>")
fmt.Println()
fmt.Println("CZ(0,1) result:")
for i, amp := range sv01 {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 1e-10 {
		fmt.Printf("  |%02b> : %+.4f\n", i, real(amp))
	}
}

fmt.Println("\nCZ(1,0) result:")
for i, amp := range sv10 {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 1e-10 {
		fmt.Printf("  |%02b> : %+.4f\n", i, real(amp))
	}
}

// Verify they are identical
identical := true
for i := range sv01 {
	if cmplx.Abs(sv01[i]-sv10[i]) > 1e-10 {
		identical = false
		break
	}
}
fmt.Printf("\nCZ(0,1) == CZ(1,0): %v\n", identical)
fmt.Println("\nCZ is symmetric because its matrix is diagonal:")
fmt.Println("  diag(1, 1, 1, -1)")
fmt.Println("Only |11> picks up a phase of -1, regardless of which qubit is 'control'.")

CZ Symmetry Demonstration
Input state: H|0> tensor X|0> = |+1>

CZ(0,1) result:
  |10> : +0.7071
  |11> : -0.7071

CZ(1,0) result:
  |10> : +0.7071
  |11> : -0.7071

CZ(0,1) == CZ(1,0): true

CZ is symmetric because its matrix is diagonal:
  diag(1, 1, 1, -1)
Only |11> picks up a phase of -1, regardless of which qubit is 'control'.


## SWAP Gate

The **SWAP** gate exchanges the states of two qubits:
- |01⟩ becomes |10⟩ (qubit 0 was on, now qubit 1 is on)
- |10⟩ becomes |01⟩
- |00⟩ and |11⟩ are unchanged

SWAP is equivalent to three consecutive CNOTs: CNOT(0,1), CNOT(1,0), CNOT(0,1).
The transpiler inserts SWAPs to route qubits on hardware with limited connectivity.

In [ ]:
%%
// SWAP demonstration: prepare |10> (qubit 1 set), SWAP -> |01> (qubit 0 set)
cSwap, _ := builder.New("swap-demo", 2).
	X(1).       // set qubit 1 -> state |10>
	SWAP(0, 1). // SWAP -> should give |01>
	Build()

fmt.Println("SWAP Circuit:")
gonbui.DisplayHTML(draw.SVG(cSwap))

sim := statevector.New(2)
sim.Evolve(cSwap)
sv := sim.StateVector()

fmt.Println("Input:  |10> (qubit 1 set)")
fmt.Println("Output statevector:")
for i, amp := range sv {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 1e-10 {
		fmt.Printf("  |%02b> : %.4f\n", i, amp)
	}
}
fmt.Println("\nSWAP exchanged qubit states: |10> -> |01>")

## Toffoli Gate (CCX) -- Reversible AND

The **Toffoli gate** (also called CCX or doubly-controlled NOT) flips the
target qubit if and only if **both** control qubits are |1>. It is the
quantum analog of the classical AND gate and is **universal for reversible
classical computation**: any classical Boolean function can be built from
Toffoli gates alone.

In our circuit `CCX(0, 1, 2)`, qubits 0 and 1 are the controls and qubit 2
is the target. In the bitstring $|q_2\,q_1\,q_0\rangle$, the controls are
the two rightmost bits. The target flips only when $q_0 = 1$ AND $q_1 = 1$.

In [ ]:
%%
// CCX (Toffoli) truth table: evolve all 8 basis states for 3 qubits
// Bitstring convention: |q2 q1 q0> where q0 and q1 are controls, q2 is target
fmt.Println("Toffoli (CCX) Truth Table  (controls=q0,q1; target=q2)")
fmt.Println("======================================================")
fmt.Println("  Input  -> Output")

for idx := 0; idx < 8; idx++ {
	b := builder.New("ccx", 3)
	// idx bit k -> qubit k
	if idx&1 == 1 {
		b.X(0)
	}
	if (idx>>1)&1 == 1 {
		b.X(1)
	}
	if (idx>>2)&1 == 1 {
		b.X(2)
	}
	b.CCX(0, 1, 2)
	c, _ := b.Build()

	sim := statevector.New(3)
	sim.Evolve(c)
	sv := sim.StateVector()

	for outIdx, amp := range sv {
		if real(amp)*real(amp)+imag(amp)*imag(amp) > 0.5 {
			flipped := ""
			if (idx>>2)&1 != (outIdx>>2)&1 {
				flipped = "  <-- target flipped"
			}
			fmt.Printf("  |%03b>  ->  |%03b>%s\n", idx, outIdx, flipped)
		}
	}
}

fmt.Println("\nThe target flips only when BOTH controls are |1>.")
fmt.Println("If q2 starts as |0>, the output q2 = q0 AND q1 (reversible AND).")

// Show the circuit
cCCXDemo, _ := builder.New("ccx", 3).X(0).X(1).CCX(0, 1, 2).Build()
fmt.Println("\nCircuit for |011> -> |111>:")
gonbui.DisplayHTML(draw.SVG(cCCXDemo))

## Fredkin Gate (CSWAP) -- Controlled SWAP

The **Fredkin gate** (CSWAP) swaps q1 and q2 if and only if the control q0
is |1>. Like the Toffoli gate, it is universal for reversible classical
computation. It is self-inverse: applying CSWAP twice returns to the
original state.

In our circuit `Apply(gate.CSWAP, 0, 1, 2)`, qubit 0 is the control and
qubits 1 and 2 are the swap targets. When q0=0, the targets are untouched;
when q0=1, the targets swap.

In [ ]:
%%
// CSWAP (Fredkin) demonstration
// Test 1: control q0=|0>, q1=|0>, q2=|1> -> should NOT swap
cNoSwap, _ := builder.New("cswap-no", 3).
	X(2).                         // set q2 -> state |100>
	Apply(gate.CSWAP, 0, 1, 2).   // control=q0=|0>, should not swap
	Build()

simNo := statevector.New(3)
simNo.Evolve(cNoSwap)
svNo := simNo.StateVector()

fmt.Println("CSWAP (Fredkin) Demonstration")
fmt.Println("=============================")
fmt.Println("\nTest 1: Control q0=0, targets q1=0, q2=1  (state |100>)")
for i, amp := range svNo {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 0.5 {
		fmt.Printf("  Result: |%03b> -- targets NOT swapped (control is off)\n", i)
	}
}

// Test 2: control q0=|1>, q1=|0>, q2=|1> -> should swap q1 and q2
cYesSwap, _ := builder.New("cswap-yes", 3).
	X(0).                          // set control q0 to |1>
	X(2).                          // set q2 to |1> -> state |101>
	Apply(gate.CSWAP, 0, 1, 2).    // control=q0=|1>, swap q1 and q2
	Build()

simYes := statevector.New(3)
simYes.Evolve(cYesSwap)
svYes := simYes.StateVector()

fmt.Println("\nTest 2: Control q0=1, targets q1=0, q2=1  (state |101>)")
for i, amp := range svYes {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 0.5 {
		fmt.Printf("  Result: |%03b> -- targets SWAPPED (q1 and q2 exchanged)\n", i)
	}
}

fmt.Println("\nCircuit for control=|1> case:")
gonbui.DisplayHTML(draw.SVG(cYesSwap))

## Controlled-U and Multi-Controlled Gates

Goqu provides flexible tools for building controlled operations:

- **`gate.Controlled(inner, nControls)`** wraps any gate with control qubits.
  The gate fires only when all control qubits are |1>.
- **`builder.Ctrl(g, controls, targets...)`** is the fluent builder version.
- **`builder.MCX(controls, target)`** is a shortcut for multi-controlled X.

For example, `gate.Controlled(gate.H, 1)` creates a controlled-Hadamard,
and `builder.Ctrl(gate.RY(pi/4), []int{0,1}, 2)` creates a doubly-controlled
RY gate.

In [7]:
%%
// Controlled-H: apply H to target only when control is |1>
fmt.Println("Controlled-H Demonstration")
fmt.Println("==========================")

// Case 1: control=|0> -- target should remain |0>
cCH0, _ := builder.New("ch-0", 2).
	Ctrl(gate.H, []int{0}, 1).
	Build()
sim0 := statevector.New(2)
sim0.Evolve(cCH0)
sv0 := sim0.StateVector()
fmt.Println("Control=|0>: target stays |0>")
for i, amp := range sv0 {
	p := real(amp)*real(amp) + imag(amp)*imag(amp)
	if p > 1e-10 {
		fmt.Printf("  |%02b> : %.4f (prob=%.4f)\n", i, amp, p)
	}
}

// Case 2: control=|1> -- target enters superposition
cCH1, _ := builder.New("ch-1", 2).
	X(0).                        // set control to |1>
	Ctrl(gate.H, []int{0}, 1).   // controlled-H on target
	Build()
sim1 := statevector.New(2)
sim1.Evolve(cCH1)
sv1 := sim1.StateVector()
fmt.Println("\nControl=|1>: target enters superposition")
for i, amp := range sv1 {
	p := real(amp)*real(amp) + imag(amp)*imag(amp)
	if p > 1e-10 {
		fmt.Printf("  |%02b> : %.4f (prob=%.4f)\n", i, amp, p)
	}
}

fmt.Println("\nCircuit (control=|1> case):")
gonbui.DisplayHTML(draw.SVG(cCH1))

// Multi-controlled RY: controls=[0,1], target=2
fmt.Println("\nMulti-Controlled RY(pi/4) with 2 controls")
fmt.Println("==========================================")
cMCRY, _ := builder.New("mc-ry", 3).
	X(0).X(1). // set both controls to |1>
	Ctrl(gate.RY(math.Pi/4), []int{0, 1}, 2).
	Build()

simMCRY := statevector.New(3)
simMCRY.Evolve(cMCRY)
svMCRY := simMCRY.StateVector()

fmt.Println("Both controls=|1>, target gets RY(pi/4):")
for i, amp := range svMCRY {
	p := real(amp)*real(amp) + imag(amp)*imag(amp)
	if p > 1e-10 {
		fmt.Printf("  |%03b> : %+.4f (prob=%.4f)\n", i, amp, p)
	}
}
fmt.Println("\nCircuit:")
gonbui.DisplayHTML(draw.SVG(cMCRY))

Controlled-H Demonstration
Control=|0>: target stays |0>
  |00> : (1.0000+0.0000i) (prob=1.0000)

Control=|1>: target enters superposition
  |01> : (0.7071+0.0000i) (prob=0.5000)
  |11> : (0.7071+0.0000i) (prob=0.5000)

Circuit (control=|1> case):


q0 
 
 q1 
 
 X 
 
 
 H

q0 
 
 q1 
 
 q2 
 
 X 
 X 
 
 
 
 RY(pi/4)


Multi-Controlled RY(pi/4) with 2 controls
Both controls=|1>, target gets RY(pi/4):
  |011> : (+0.9239+0.0000i) (prob=0.8536)
  |111> : (+0.3827+0.0000i) (prob=0.1464)

Circuit:


In [ ]:
%%
// MCX: Multi-controlled X with 3 controls on a 4-qubit circuit
fmt.Println("MCX (Multi-Controlled X) with 3 Controls")
fmt.Println("=========================================")

// All controls |1> -> target flips
cMCX, _ := builder.New("mcx", 4).
	X(0).X(1).X(2).                 // set all 3 controls to |1>
	MCX([]int{0, 1, 2}, 3).         // multi-controlled X on target q3
	Build()

simMCX := statevector.New(4)
simMCX.Evolve(cMCX)
svMCX := simMCX.StateVector()

fmt.Println("Input: |0111> (3 controls q0,q1,q2 = |1>, target q3 = |0>)")
for i, amp := range svMCX {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 0.5 {
		fmt.Printf("  Output: |%04b> -- target flipped to |1>\n", i)
	}
}

// Only 2 of 3 controls |1> -> target stays
cMCX2, _ := builder.New("mcx-partial", 4).
	X(0).X(1).                       // only 2 controls are |1>
	MCX([]int{0, 1, 2}, 3).          // target should NOT flip
	Build()

simMCX2 := statevector.New(4)
simMCX2.Evolve(cMCX2)
svMCX2 := simMCX2.StateVector()

fmt.Println("\nInput: |0011> (only 2 of 3 controls = |1>)")
for i, amp := range svMCX2 {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 0.5 {
		fmt.Printf("  Output: |%04b> -- target unchanged (not all controls active)\n", i)
	}
}

fmt.Println("\nCircuit for 3-controlled X:")
gonbui.DisplayHTML(draw.SVG(cMCX))

## Custom Unitary Gates

When the standard gate library does not have the gate you need, Goqu lets you
create one from a raw unitary matrix using `gate.MustUnitary(name, matrix)`.
The matrix must be:
- **Square**: 2x2 (1 qubit), 4x4 (2 qubits), or 8x8 (3 qubits)
- **Unitary**: U dagger U = I (verified automatically)
- **Flat row-major**: `[]complex128` of length 4, 16, or 64

Let's create a custom sqrt(X) gate whose matrix is:

$$\sqrt{X} = \frac{1}{2}\begin{pmatrix} 1+i & 1-i \\ 1-i & 1+i \end{pmatrix}$$

Applying it twice should equal the X gate.

In [9]:
%%
// Custom unitary: sqrt(X) gate
sqrtXMatrix := []complex128{
	complex(0.5, 0.5), complex(0.5, -0.5),
	complex(0.5, -0.5), complex(0.5, 0.5),
}
sqrtX := gate.MustUnitary("sqrtX", sqrtXMatrix)

fmt.Println("Custom sqrt(X) Gate")
fmt.Println("===================")
fmt.Printf("Name:   %s\n", sqrtX.Name())
fmt.Printf("Qubits: %d\n", sqrtX.Qubits())
fmt.Println("Matrix:", sqrtX.Matrix())

// Apply sqrt(X) to |0>
cSqrtX, _ := builder.New("sqrtx", 1).Apply(sqrtX, 0).Build()
simSX := statevector.New(1)
simSX.Evolve(cSqrtX)
svSX := simSX.StateVector()
fmt.Println("\nsqrt(X)|0> =", svSX)

// Apply sqrt(X) twice -- should equal X|0> = |1>
cSqrtX2, _ := builder.New("sqrtx2", 1).
	Apply(sqrtX, 0).
	Apply(sqrtX, 0).
	Build()
simSX2 := statevector.New(1)
simSX2.Evolve(cSqrtX2)
svSX2 := simSX2.StateVector()
fmt.Println("sqrt(X)^2 |0> =", svSX2)

// Verify: sqrt(X)^2 = X (up to global phase)
p0 := real(svSX2[0])*real(svSX2[0]) + imag(svSX2[0])*imag(svSX2[0])
p1 := real(svSX2[1])*real(svSX2[1]) + imag(svSX2[1])*imag(svSX2[1])
fmt.Printf("\nP(|0>) = %.4f, P(|1>) = %.4f\n", p0, p1)
fmt.Println("sqrt(X) applied twice gives |1> (= X|0>):", p1 > 0.999)

fmt.Println("\nCircuit:")
gonbui.DisplayHTML(draw.SVG(cSqrtX2))

Custom sqrt(X) Gate
Name:   sqrtX
Qubits: 1
Matrix: [(0.5+0.5i) (0.5-0.5i) (0.5-0.5i) (0.5+0.5i)]

sqrt(X)|0> = [(0.5+0.5i) (0.5-0.5i)]
sqrt(X)^2 |0> = [(0+0i) (1+0i)]

P(|0>) = 0.0000, P(|1>) = 1.0000
sqrt(X) applied twice gives |1> (= X|0>): true

Circuit:


q0 
 
 sqrtX 
 sqrtX

## Gate Decomposition

Real quantum hardware can only execute a small set of **native gates** (e.g.,
CNOT + single-qubit rotations on IBM, or CZ + single-qubit on Google). Any
higher-level gate must be **decomposed** into these primitives.

For example, the Toffoli gate (CCX) requires 6 CNOTs and several single-qubit
gates when decomposed for hardware. The Goqu `Gate.Decompose(qubits)` method
returns such a decomposition when available.

Let's examine the matrices of multi-qubit gates to understand their structure
and verify that controlled gates are identity except in the subspace where
all controls are |1>.

In [10]:
%%
// Examine the structure of multi-qubit gate matrices
fmt.Println("Gate Matrix Structure")
fmt.Println("=====================")

// Helper to print a gate matrix
printMatrix := func(name string, g gate.Gate) {
	m := g.Matrix()
	dim := 1 << g.Qubits()
	fmt.Printf("\n%s (%d-qubit, %dx%d matrix):\n", name, g.Qubits(), dim, dim)
	for r := 0; r < dim; r++ {
		fmt.Print("  [")
		for c := 0; c < dim; c++ {
			v := m[r*dim+c]
			if cmplx.Abs(v) < 1e-10 {
				fmt.Print("  .  ")
			} else if imag(v) == 0 {
				fmt.Printf("%5.1f", real(v))
			} else {
				fmt.Printf("%5.1f", v)
			}
		}
		fmt.Println(" ]")
	}
}

printMatrix("CNOT", gate.CNOT)
printMatrix("CZ", gate.CZ)
printMatrix("SWAP", gate.SWAP)

// Show that CCX is identity except the bottom-right 2x2 block
fmt.Println("\nCCX (Toffoli) structure:")
fmt.Println("The 8x8 matrix is identity except rows/cols 6-7,")
fmt.Println("where the X gate acts (swaps |110> and |111>):")
printMatrix("CCX", gate.CCX)

// Demonstrate gate.Controlled builds the right matrix
ch := gate.Controlled(gate.H, 1)
fmt.Printf("\nControlled(H, 1) name: %s, qubits: %d\n", ch.Name(), ch.Qubits())
printMatrix("Controlled-H", ch)

Gate Matrix Structure

CNOT (2-qubit, 4x4 matrix):
  [  1.0  .    .    .   ]
  [  .    1.0  .    .   ]
  [  .    .    .    1.0 ]
  [  .    .    1.0  .   ]

CZ (2-qubit, 4x4 matrix):
  [  1.0  .    .    .   ]
  [  .    1.0  .    .   ]
  [  .    .    1.0  .   ]
  [  .    .    .   -1.0 ]

SWAP (2-qubit, 4x4 matrix):
  [  1.0  .    .    .   ]
  [  .    .    1.0  .   ]
  [  .    1.0  .    .   ]
  [  .    .    .    1.0 ]

CCX (Toffoli) structure:
The 8x8 matrix is identity except rows/cols 6-7,
where the X gate acts (swaps |110> and |111>):

CCX (3-qubit, 8x8 matrix):
  [  1.0  .    .    .    .    .    .    .   ]
  [  .    1.0  .    .    .    .    .    .   ]
  [  .    .    1.0  .    .    .    .    .   ]
  [  .    .    .    1.0  .    .    .    .   ]
  [  .    .    .    .    1.0  .    .    .   ]
  [  .    .    .    .    .    1.0  .    .   ]
  [  .    .    .    .    .    .    .    1.0 ]
  [  .    .    .    .    .    .    1.0  .   ]

Controlled(H, 1) name: C1-H, qubits: 2

Controlled-H (2-qubit,

## Predict, Then Verify

**Question:** What does CNOT do to the state $|+0\rangle$? (That is, H on qubit 0, then CNOT with control=q0, target=q1.)

**Pause and predict** before reading further. The state $|+0\rangle = (|00\rangle + |01\rangle)/\sqrt{2}$ has qubit 0 in superposition and qubit 1 in $|0\rangle$. Remember that in our bitstring notation $|q_1\,q_0\rangle$, qubit 0 is the rightmost bit.

*Hint: The CNOT flips q1 when q0=1. Think about what happens to each term in the superposition separately.*

In [11]:
%%
// Predict-then-verify: CNOT on |+0> creates a Bell state
cBell, _ := builder.New("bell", 2).
	H(0).        // |0> -> |+> on q0
	CNOT(0, 1).  // entangle
	Build()

fmt.Println("Circuit: H(0) then CNOT(0,1)")
gonbui.DisplayHTML(draw.SVG(cBell))

simBell := statevector.New(2)
simBell.Evolve(cBell)
svBell := simBell.StateVector()

fmt.Println("Statevector:")
for i, amp := range svBell {
	if real(amp)*real(amp)+imag(amp)*imag(amp) > 1e-10 {
		fmt.Printf("  |%02b> : %+.4f\n", i, real(amp))
	}
}

fmt.Println("\nPrediction confirmed: CNOT|+0> = (|00> + |11>)/sqrt(2) = Bell state |Phi+>")

// Measure 1000 times to confirm only |00> and |11> appear
cBellM, _ := builder.New("bell-measure", 2).H(0).CNOT(0, 1).MeasureAll().Build()
simM := statevector.New(2)
counts, _ := simM.Run(cBellM, 1000)

fmt.Println("\nMeasurement results (1000 shots):")
for outcome, count := range counts {
	fmt.Printf("  |%s> : %d (%.1f%%)\n", outcome, count, float64(count)/10.0)
}

svg := viz.Histogram(counts, viz.WithTitle("Bell State |Phi+> from CNOT|+0>"))
gonbui.DisplayHTML(svg)

Circuit: H(0) then CNOT(0,1)
Statevector:
  |00> : +0.7071
  |11> : +0.7071

Prediction confirmed: CNOT|+0> = (|00> + |11>)/sqrt(2) = Bell state |Phi+>

Measurement results (1000 shots):
  |00> : 473 (47.3%)
  |11> : 527 (52.7%)


q0 
 
 q1 
 
 H

Bell State |Phi+> from CNOT|+0> 
 
 
 
 0 
 
 100 
 
 200 
 
 300 
 
 400 
 
 500 
 
 600 
 
 00 
 
 11

## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Is CZ(0,1) the same as CZ(1,0)? Why or why not?
2. How many CNOT gates does a SWAP gate decompose into?
3. What state does CNOT|+0⟩ produce, and why is it significant?

## Exercises

### Exercise 1: 4-Qubit Controlled-RY

Use `builder.Ctrl` to create a 3-controlled RY(pi/4) gate on a 4-qubit
circuit. Set all three control qubits to |1> and verify that the target
qubit receives the RY rotation.

In [12]:
%%
// Exercise 1: 3-Controlled RY(pi/4) on 4 Qubits
//
// TODO: Build a 4-qubit circuit where:
//   1. Set all 3 control qubits (q0, q1, q2) to |1> using X gates
//   2. Apply a 3-controlled RY(pi/4) on the target qubit (q3)
//      using builder.Ctrl(gate.RY(math.Pi/4), []int{0, 1, 2}, 3)
//   3. Inspect the statevector to verify the rotation was applied
//
// Hint: The target should have amplitudes cos(pi/8) and sin(pi/8)

// cEx1, _ := builder.New("ctrl-ry", 4).
// 	/* TODO: Set controls to |1> */
// 	/* TODO: Apply 3-controlled RY(pi/4) */
// 	Build()
//
// fmt.Println("Exercise 1: 3-Controlled RY(pi/4) on 4 Qubits")
// fmt.Println("===============================================")
// fmt.Println("Circuit:")
// fmt.Println(draw.String(cEx1))
//
// simEx1 := statevector.New(4)
// simEx1.Evolve(cEx1)
// svEx1 := simEx1.StateVector()
//
// fmt.Println("Statevector (non-zero amplitudes):")
// for i, amp := range svEx1 {
// 	p := real(amp)*real(amp) + imag(amp)*imag(amp)
// 	if p > 1e-10 {
// 		fmt.Printf("  |%04b> : %+.4f (prob=%.4f)\n", i, amp, p)
// 	}
// }
//
// fmt.Printf("\nExpected target amplitudes: cos(pi/8)=%.4f, sin(pi/8)=%.4f\n",
// 	math.Cos(math.Pi/8), math.Sin(math.Pi/8))

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


In [13]:
%%
// Exercise 2: Custom Unitary Verification
//
// TODO: Create a custom phase-shift gate P(pi/3) = diag(1, e^{i*pi/3})
// and verify that it is unitary (U dagger U = I).
//
// Steps:
//   1. Define the 2x2 matrix: {1, 0, 0, cmplx.Exp(complex(0, phi))}
//   2. Use gate.Unitary("P(pi/3)", matrix) to create the gate
//   3. Manually compute U dagger U and check it equals the identity
//   4. (Bonus) Try creating a non-unitary gate to see the error
//
// Hint: (U dagger)[i][k] = conj(U[k][i]) for the conjugate transpose

// phi := math.Pi / 3
// customMatrix := []complex128{
// 	1, 0,
// 	0, cmplx.Exp(complex(0, phi)),
// }
//
// customGate, err := gate.Unitary("P(pi/3)", customMatrix)
// if err != nil {
// 	fmt.Println("Error: matrix is not unitary!", err)
// } else {
// 	fmt.Println("Gate created successfully (matrix IS unitary).")
// 	fmt.Printf("Name: %s, Qubits: %d\n", customGate.Name(), customGate.Qubits())
// }
//
// // TODO: Manually verify U dagger U = I
// // m := customGate.Matrix()
// // dim := 2
// // for i := 0; i < dim; i++ {
// //     for j := 0; j < dim; j++ {
// //         var sum complex128
// //         for k := 0; k < dim; k++ {
// //             sum += cmplx.Conj(m[i*dim+k]) * m[j*dim+k]
// //         }
// //         fmt.Printf("  (U dagger U)[%d][%d] = %+.6f\n", i, j, sum)
// //     }
// // }
//
// // TODO: Try creating a non-unitary gate
// // badMatrix := []complex128{1, 1, 0, 1}
// // _, err = gate.Unitary("bad", badMatrix)
// // fmt.Println("Rejected:", err)

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


## Key Takeaways

1. **CNOT** is the fundamental two-qubit gate: it flips the target when the
   control is |1>. Together with single-qubit gates, it forms a universal
   gate set.

2. **CZ** is symmetric in its qubits -- swapping control and target gives the
   same result. This makes it convenient for hardware with symmetric couplings.

3. **SWAP** exchanges two qubit states and is equivalent to three CNOTs.
   Transpilers insert SWAPs for qubit routing on real hardware.

4. **Toffoli (CCX)** is a reversible AND gate: the target flips only when
   both controls are |1>. Universal for reversible classical computation.

5. **Fredkin (CSWAP)** is a controlled SWAP: the two targets swap only when
   the control is |1>. Also universal for reversible computation.

6. **Controlled-U** and **multi-controlled gates** generalize the control
   pattern to arbitrary gates. Use `gate.Controlled(g, n)` or
   `builder.Ctrl(g, controls, targets...)` in Goqu.

7. **Custom unitary gates** can be created from raw matrices using
   `gate.MustUnitary(name, matrix)`. Goqu automatically verifies unitarity.

8. **Gate decomposition** is essential for real hardware: multi-qubit gates
   must be broken into native 1Q and 2Q operations.

9. **Superposition + controlled operation = entanglement**: applying CNOT to
   |+0> creates the Bell state, the simplest entangled state.